# NB6 — Final validation-AUC training V5

Đây không còn là ranking-loss experiment.

Training path được chọn từ các diagnostic đã hoàn tất:

- **BCEWithLogitsLoss only**
- **standard sample-level shuffled batches**
- **full FP32 training**
- **FP32 validation**
- learned category embedding vẫn được giữ, nhưng category vectors được scale về norm xấp xỉ 1
- category re-initialization diễn ra **sau khi các MLP đã được construct**, để không làm lệch downstream MLP RNG initialization
- `max_epochs = 60`
- `early_stopping_patience = 10`
- **không cho early stop trước epoch 30**
- checkpoint selection chỉ bằng **validation ROC-AUC**
- **không đọc test split**

Notebook sử dụng `configs/scorer_type_aware_pairwise_v1_val_auc.yaml`.

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys

REPO_URL = "https://github.com/ThinhTran2208/opisoverated.git"
REPO_ROOT = Path("/content/opisoverated")

if not REPO_ROOT.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_ROOT)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_ROOT), "pull", "--ff-only"], check=True)

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from google.colab import drive
drive.mount("/content/drive")

ARTIFACT_ROOT = Path("/content/drive/MyDrive/ML_Final")
os.environ["FASHION_ARTIFACT_ROOT"] = str(ARTIFACT_ROOT)
os.environ["FASHION_EMBEDDING_CACHE"] = str(
    ARTIFACT_ROOT / "fashionclip_item_embeddings.pt"
)
os.environ["FASHION_EMBEDDING_MANIFEST"] = str(
    ARTIFACT_ROOT / "embedding_manifest_v1.json"
)
os.environ["FASHION_CORE7_DIR"] = str(
    ARTIFACT_ROOT / "polyvore_core7_v2" / "core7_drop_v2"
)
os.environ["FASHION_SCORER_READY_DIR"] = str(
    ARTIFACT_ROOT / "polyvore_core7_v2" / "scorer_ready_v2"
)

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "pyyaml"],
    check=True,
)

HEAD = subprocess.check_output(
    ["git", "-C", str(REPO_ROOT), "rev-parse", "HEAD"],
    text=True,
).strip()

print("Git HEAD:", HEAD)
print("Artifact root:", ARTIFACT_ROOT)

In [ ]:
# Regression checks for the source changes used by this final run.
subprocess.run(
    [
        sys.executable,
        "-m",
        "unittest",
        "tests.test_scorer_model",
        "tests.test_scorer_init_order",
        "tests.test_scorer_train",
    ],
    cwd=REPO_ROOT,
    check=True,
)

print("SCORER REGRESSION TESTS: PASS")

In [ ]:
import yaml
import torch

from src.data.runtime_paths import load_runtime_paths
from src.scorer.checkpoint import build_runtime_provenance, load_checkpoint
from src.scorer.model import TypeAwarePairwiseScorer
from src.scorer.train import (
    build_train_valid_loaders,
    evaluate_epoch,
    fit_scorer,
    seed_everything,
    validate_s3_config,
)

CONFIG_PATH = (
    REPO_ROOT
    / "configs"
    / "scorer_type_aware_pairwise_v1_val_auc.yaml"
)

with CONFIG_PATH.open("r", encoding="utf-8") as f:
    config = yaml.safe_load(f)

validate_s3_config(config)

training = config["training"]
assert training["mixed_precision"] is False
assert training["max_epochs"] == 60
assert training["early_stopping_patience"] == 10
assert training["early_stopping_min_epochs"] == 30
assert training["seed"] == 42
assert config["selection"]["primary_metric"] == "roc_auc"

paths = load_runtime_paths(repo_root=REPO_ROOT)
provenance = build_runtime_provenance(paths, REPO_ROOT)

assert provenance["git_tree_clean"] is True
assert provenance["git_commit"] == HEAD

print("Training config:")
print(json.dumps(training, indent=2))
print("CONFIG / PROVENANCE: PASS")

In [ ]:
# Build loaders exactly once, fresh for this final run.
loaders = build_train_valid_loaders(
    paths,
    config,
    num_workers=0,
)

train_dataset = loaders["datasets"]["train"]
valid_dataset = loaders["datasets"]["valid"]
train_loader = loaders["train_loader"]
valid_loader = loaders["valid_loader"]

assert len(train_dataset) == 30918
assert len(valid_dataset) == 2284
assert len(train_dataset.pair_families) == 15459
assert len(valid_dataset.pair_families) == 1142
assert len(train_loader) == 121
assert len(valid_loader) == 9

print("Train samples/families:", len(train_dataset), len(train_dataset.pair_families))
print("Valid samples/families:", len(valid_dataset), len(valid_dataset.pair_families))
print("Train/valid batches:", len(train_loader), len(valid_loader))
print("FRESH STANDARD LOADERS: PASS")

In [ ]:
SEED = int(training["seed"])
seed_everything(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
assert device.type == "cuda", "Use a GPU runtime; training remains FP32."

model = TypeAwarePairwiseScorer.from_config(config).to(device)

with torch.no_grad():
    category_weights = model.category_embedding.weight.detach()
    category_norm_mean = float(category_weights[1:].norm(dim=1).mean())
    pad_norm = float(category_weights[0].norm())

assert model.category_embedding_init_policy == "post_mlp_scale_preserving"
assert 0.7 < category_norm_mean < 1.3
assert pad_norm == 0.0

print("GPU:", torch.cuda.get_device_name(0))
print("Parameters:", sum(p.numel() for p in model.parameters()))
print("Category init policy:", model.category_embedding_init_policy)
print("Mean real-category norm:", category_norm_mean)
print("PAD norm:", pad_norm)
print("Training precision: FP32")
print("MODEL INIT: PASS")

In [ ]:
RUN_DIR = (
    paths.artifact_root
    / "scorer_runs"
    / "type_aware_pairwise_v1"
    / "final_val_auc_v5_seed42"
)

if RUN_DIR.exists() and any(RUN_DIR.iterdir()):
    raise RuntimeError(
        f"Run directory already contains outputs: {RUN_DIR}\n"
        "Use a new run name instead of overwriting the final run."
    )

result = fit_scorer(
    model,
    train_loader,
    valid_loader,
    config=config,
    checkpoint_dir=RUN_DIR,
    provenance=provenance,
    device=device,
)

assert result["mixed_precision_active"] is False
assert result["training_precision"] == "fp32"
assert result["validation_precision"] == "fp32"
assert result["epochs_ran"] >= 30

print()
print("Best epoch:", result["best_epoch"])
print("Best validation ROC-AUC:", result["best_valid_roc_auc"])
print("Epochs ran:", result["epochs_ran"])
print("Stopped early:", result["stopped_early"])
print("Best checkpoint:", result["best_checkpoint"])

In [ ]:
# Re-load best.pt and evaluate validation once in FP32.
best_model = TypeAwarePairwiseScorer.from_config(config).to(device)

best_payload = load_checkpoint(
    result["best_checkpoint"],
    model=best_model,
    map_location=device,
    current_provenance=provenance,
)

best_model.eval()

criterion = torch.nn.BCEWithLogitsLoss()
best_valid = evaluate_epoch(
    best_model,
    valid_loader,
    criterion=criterion,
    device=device,
)

assert best_payload["epoch"] == result["best_epoch"]
assert best_valid["sample_count"] == 2284
assert best_valid["paired_family_count"] == 1142

print("FINAL BEST VALIDATION")
for key, value in best_valid.items():
    print(f"{key:26s}: {value}")

In [ ]:
# Persist a compact final summary beside the checkpoints.
summary = {
    "git_commit": HEAD,
    "run_dir": str(RUN_DIR),
    "seed": SEED,
    "training_precision": result["training_precision"],
    "validation_precision": result["validation_precision"],
    "max_epochs": training["max_epochs"],
    "early_stopping_patience": training["early_stopping_patience"],
    "early_stopping_min_epochs": training["early_stopping_min_epochs"],
    "best_epoch": result["best_epoch"],
    "epochs_ran": result["epochs_ran"],
    "best_valid_roc_auc": best_valid["roc_auc"],
    "best_valid_fitb_2way": best_valid["fitb_2way"],
    "best_valid_mean_margin": best_valid["mean_logit_margin"],
    "best_valid_median_margin": best_valid["median_logit_margin"],
    "category_embedding_init_policy": model.category_embedding_init_policy,
    "category_norm_mean_at_init": category_norm_mean,
}

with (RUN_DIR / "final_summary.json").open("w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

with (RUN_DIR / "history.json").open("w", encoding="utf-8") as f:
    json.dump(result["history"], f, indent=2)

print(json.dumps(summary, indent=2))
print()
print("TEST SPLIT WAS NOT LOADED.")
print("NB6 FINAL VALIDATION-AUC V5: COMPLETE")

In [ ]:
# Optional learning-curve view. No model selection is performed here.
import matplotlib.pyplot as plt

history = result["history"]
epochs = [row["epoch"] for row in history]
aucs = [row["valid_roc_auc"] for row in history]

plt.figure(figsize=(8, 4))
plt.plot(epochs, aucs)
plt.xlabel("Epoch")
plt.ylabel("Validation ROC-AUC")
plt.title("Validation ROC-AUC")
plt.grid(True, alpha=0.25)
plt.show()